<a href="https://colab.research.google.com/github/dhiyasalmas/Machine_Learning/blob/main/MINDSTEM_ImageClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pelatihan Pengembangan AI - MINDSTEM

Pada notebook ini, kita akan membuat sebuah aplikasi sederhana untuk mendeteksi objek pada gambar menggunakan model dari [HuggingFace](https://huggingface.co/) (atau [HF](https://huggingface.co/)). HF adalah platform untuk membangun dan berbagi model open-source AI. Kita akan menggunakan Google Colab untuk membuat aplikasi ini dan men-deploy prototype-nya menggunakan [Gradio](https://www.gradio.app/).


# Langkah 1: Instalasi

Kita akan menggunakan beberapa library berikut:

- `transformers` untuk memuat model dari HF.
- `gradio` untuk membangun aplikasi web.
- `timm` untuk memuat model dari HF.

In [ ]:
!pip install transformers
!pip install gradio
!pip install timm

# Langkah 2: Memuat Model

Kita akan memuat model dari HF menggunakan library `transformers`, khususnya menggunakan fungsi `pipeline`.

In [ ]:
from transformers import pipeline

### Tentang `pipeline`

- Istilah `pipeline` di sini merujuk ke sebuah objek yang sudah terintegrasi dengan model dan task yang akan kita gunakan.
- Biasanya, tiap model punya spesifikasi input dan output yang berbeda-beda (seringkali kompleks).
- Dengan menggunakan `pipeline`, kita tidak perlu khawatir tentang hal ini.
- Input yang kita berikan akan diubah menjadi format yang dibutuhkan oleh model yang kita pilih.

### Pilih model

Mari kita lihat ada model apa saja yang tersedia di [HF](https://huggingface.co/models) untuk [task object detection](https://huggingface.co/models?pipeline_tag=object-detection&sort=downloads) ini.

Kita coba pakai `DETR` yang merupakan model yang cukup populer untuk task ini. Cara memuat model ini cukup mudah, kita hanya perlu menyebutkan nama model dan task yang akan kita gunakan.

In [ ]:
pipe = pipeline("object-detection", model="facebook/detr-resnet-50")

Proses di atas akan memuat model dari HF ke dalam memory kita (sehingga model di-run di Colab, bukan di server HF).

Ini juga berarti, kalau kita run notebook ini di komputer lokal kita, maka model akan di-download ke komputer lokal kita, dan semua proses komputasi (inference) akan dijalankan di komputer lokal kita.

# Langkah 3: Mencoba model

Kita coba pakai model ini untuk mendeteksi gambar. Silakan download (save ke file) gambar yang ingin Anda deteksi. Misalnya:

- [Siswa](https://asiatoday.id/wp-content/uploads/2020/06/PicsArt_06-07-01.28.38.png)
- [Botol](https://c.pxhere.com/photos/d2/95/pollution_garbage_thrown_away_waste_disposed_of_environment_bottle_plastic-962008.jpg!)

Mari kita bikin fungsi untuk memuat gambar dari URL.

In [ ]:
import requests
from PIL import Image

def load_image_from_url(url):
    return Image.open(requests.get(url, stream=True).raw)

Serta fungsi untuk menampilkan gambar, bounding box, dan label.

In [ ]:
import io
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def display_image_with_bounding_boxes(image, bounding_boxes, labels):
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(image)
    for box, label in zip(bounding_boxes, labels):
        xmin = box['xmin']
        ymin = box['ymin']
        xmax = box['xmax']
        ymax = box['ymax']
        w = xmax-xmin
        h = ymax-ymin
        ax.add_patch(plt.Rectangle((xmin, ymin),w,h, fill=False, color="green", linewidth=2))
        ax.text(xmin, ymin, f"{label}",color='blue')
    plt.axis("off")


Mari kita coba untuk gambar botol.

In [ ]:
image_url = "https://c.pxhere.com/photos/d2/95/pollution_garbage_thrown_away_waste_disposed_of_environment_bottle_plastic-962008.jpg!d"
img = load_image_from_url(image_url)
img

Kita coba tampilkan outputnya.

In [ ]:
output = pipe(img)

In [ ]:
output

Hasil dari model ini adalah sebuah `list` of `dict`, yang masing-masing `dict` merepresentasikan sebuah bounding box dan label.

Mari kita format outputnya untuk di-plot.

In [ ]:
boxes = [output[0]['box']]
labels =  [output[0]['label']]

display_image_with_bounding_boxes(img, boxes, labels)

Bravo!

### Perbaiki Display

Mari kita edit fungsi display-nya supaya outputnya lebih menarik.

In [ ]:
def render_results_in_image(in_pil_img, in_results):
    plt.figure(figsize=(16, 10))
    plt.imshow(in_pil_img)

    ax = plt.gca()

    for prediction in in_results:

        x, y = prediction['box']['xmin'], prediction['box']['ymin']
        w = prediction['box']['xmax'] - prediction['box']['xmin']
        h = prediction['box']['ymax'] - prediction['box']['ymin']

        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, color="green", linewidth=2))
        ax.text(x, y, f"{prediction['label']}: {round(prediction['score']*100, 1)}%", color='red')

    plt.axis("off")

    # Save the modified image to a BytesIO object
    img_buf = io.BytesIO()
    plt.savefig(img_buf, format='png', bbox_inches='tight', pad_inches=0)
    img_buf.seek(0)
    modified_image = Image.open(img_buf)

    # Close the plot to prevent it from being displayed
    plt.close()

    return modified_image

In [ ]:
image_url = "https://asiatoday.id/wp-content/uploads/2020/06/PicsArt_06-07-01.28.38.png"
img = load_image_from_url(image_url)
img

In [ ]:
#Tahapan utama
output = pipe(img) #prosess img
render_results_in_image(img, output) #display hasil

# Langkah 4: Membuat aplikasi

In [ ]:
# Load library yg dibutuhkan
import os
import gradio as gr

In [ ]:
# Gabung tahapan utama menjadi satu fungsi
def get_pipeline_prediction(pil_image):
    pipe_output = pipe(pil_image)
    processed_image = render_results_in_image(pil_image, pipe_output)
    return processed_image

In [ ]:
# Set up interface app-nya
demo = gr.Interface(
    fn=get_pipeline_prediction,
    inputs=gr.Image(label="Gambar input", type="pil"),
    outputs=gr.Image(label="Display output", type="pil")
)

In [ ]:
# Launch
demo.launch(share=True)

Untuk menutup app-nya:

In [ ]:
demo.close()

# Langkah 5: Deploy ke HF

Mari kita deploy app-nya ke Hugging Face. Langkah-langkahnya adalah sebagai berikut:

1. Buat `space` di HF.
2. Buat file teks `requirements.txt` yang berisi library yang dibutuhkan.
3. Buat file Python `app.py` yang berisi kode untuk menjalankan app.

```
# requirements.txt
transformers
matplotlib
gradio
timm

```

```
# app.py
import io
import os
import gradio as gr
from transformers import pipeline
import matplotlib.pyplot as plt
from PIL import Image

pipe = pipeline("object-detection", model="facebook/detr-resnet-50")

def get_pipeline_prediction(pil_image):
    pipe_output = pipe(pil_image)
    processed_image = render_results_in_image(pil_image, pipe_output)
    return processed_image

def render_results_in_image(in_pil_img, in_results):
    plt.figure(figsize=(16, 10))
    plt.imshow(in_pil_img)

    ax = plt.gca()

    for prediction in in_results:

        x, y = prediction['box']['xmin'], prediction['box']['ymin']
        w = prediction['box']['xmax'] - prediction['box']['xmin']
        h = prediction['box']['ymax'] - prediction['box']['ymin']

        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, color="green", linewidth=2))
        ax.text(x, y, f"{prediction['label']}: {round(prediction['score']*100, 1)}%", color='red')

    plt.axis("off")

    # Save the modified image to a BytesIO object
    img_buf = io.BytesIO()
    plt.savefig(img_buf, format='png', bbox_inches='tight', pad_inches=0)
    img_buf.seek(0)
    modified_image = Image.open(img_buf)

    # Close the plot to prevent it from being displayed
    plt.close()

    return modified_image


# Set up interface app-nya
demo = gr.Interface(
    fn=get_pipeline_prediction,
    inputs=gr.Image(label="Gambar input", type="pil"),
    outputs=gr.Image(label="Display output", type="pil")
)

# Launch
demo.launch(share=True)
```

In [ ]:
# app.py
import io
import os
import gradio as gr
from transformers import pipeline
import matplotlib.pyplot as plt
from PIL import Image

pipe = pipeline("object-detection", model="facebook/detr-resnet-50")

def get_pipeline_prediction(pil_image):
    pipe_output = pipe(pil_image)
    processed_image = render_results_in_image(pil_image, pipe_output)
    return processed_image

def render_results_in_image(in_pil_img, in_results):
    plt.figure(figsize=(16, 10))
    plt.imshow(in_pil_img)

    ax = plt.gca()

    for prediction in in_results:

        x, y = prediction['box']['xmin'], prediction['box']['ymin']
        w = prediction['box']['xmax'] - prediction['box']['xmin']
        h = prediction['box']['ymax'] - prediction['box']['ymin']

        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, color="green", linewidth=2))
        ax.text(x, y, f"{prediction['label']}: {round(prediction['score']*100, 1)}%", color='red')

    plt.axis("off")

    # Save the modified image to a BytesIO object
    img_buf = io.BytesIO()
    plt.savefig(img_buf, format='png', bbox_inches='tight', pad_inches=0)
    img_buf.seek(0)
    modified_image = Image.open(img_buf)

    # Close the plot to prevent it from being displayed
    plt.close()

    return modified_image


# Set up interface app-nya
demo = gr.Interface(
    fn=get_pipeline_prediction,
    inputs=gr.Image(label="Gambar input", type="pil"),
    outputs=gr.Image(label="Display output", type="pil")
)

# Launch
demo.launch(share=True)